## Is this trading strategy profitable?

**Start by creating a parameters array, net PnL, and outcome vectors**


In [1]:
import numpy as np

def create_strategy_params(p_win, win_amount, loss_amountm, cost):
    #  Args:
    #p_win: Probability of winning (0 to 1)
    #win_amount: Amount gained on a win
    #loss_amount: Amount lost on a loss
    #cost: Fixed cost per trade

    return np.array([p_win, win_amount, loss_amountm, cost])

def compute_trade_pnl(outcome, params):    
    #Args:
    #outcome: 1 for win, 0 for loss
    #params: numpy array [p_win, win_amount, loss_amount, cost]
    win_amount = params[1]
    loss_amount = params[2]
    cost = params[3]

    if outcome == 1:
        return win_amount - cost
    else:
        return -loss_amount - cost

def build_outcome_vectors(params):
    #Args:
    #params: numpy array [p_win, win_amount, loss_amount, cost]
    p_win = params[0]
    win_amount = params[1]
    loss_amount = params[2]
    cost = params[3]

    probabilities =np.array([p_win,1-p_win])
    net_payoffs=np.array([win_amount - cost, -loss_amount-cost])

    return probabilities, net_payoffs
    

## Exoected Value

Calculate expected value per each trade

In [2]:
import numpy as np

def expected_value(probs, payoffs):
    #Args:
    #probs: numpy array of probabilities (must sum to 1)
    #payoffs: numpy array of corresponding payoffs
    return np.dot(probs, payoffs)

def ev_from_params(params):
    #  Args:
    # params: numpy array [p_win, win_amount, loss_amount, cost]
    p_win = params[0]
    win_amount = params[1]
    loss_amount = params[2]
    cost = params[3]
    ev = p_win * win_amount - (1 - p_win) * loss_amount - cost
    return ev

def batch_ev(params_matrix):
    #Args:
    #params_matrix: 2D array where each row is [P_win, win, loss, cost]
    #shape_: (n_strategies, 4)
    p_win = params_matrix[:, 0]
    win_amount = params_matrix[:, 1]
    loss_amount = params_matrix[:, 2]
    cost = params_matrix[:, 3]
    ev = p_win * win_amount - (1 - p_win) * loss_amount - cost
    return ev

## Break-Even probability

What win probability do I need to turn break-even?

In [3]:
import numpy as np

def break_even_prob(win_amount, loss_amount, cost):
    #Args:
    #win_amount: Amount gained on a win
    #loss_amount: Amount lost on a loss
    #cost: Fixed cost per trade
    return (loss_amount+cost)/(win_amount+loss_amount)

def break_even_win(p_win, loss_amount, cost):
    #Args:
    #p_win: Probability of winning
    #loss_amount: Amount lost on a loss
    #cost: Cost per trade
    return ((1 - p_win) * loss_amount + cost) / p_win

def margin_above_breakeven(params):
    #Args:
    #params: numpy array [p_win, win_amount, loss_amount, cost]
    p_win = params[0]
    win_amount = params[1]
    loss_amount = params[2]
    cost = params[3]

    break_even_p = (loss_amount + cost) / (win_amount + loss_amount)
    margin = p_win - break_even_p
    return margin

## A very simple trade simulator

In [ ]:
import numpy as np

def simulate_outcomes(p_win, n_trades, seed=None):
    #Args:
    #p_win: Probability of winning each trade
    #n_trades: Number of trades to simulate
    #seed: Random seed for reproducibility
    rng=np.random.default_rng(seed)
    random_vals=rng.random(n_trades)
    outcomes=(random_vals<p_win).astype(int)
    return outcomes

import numpy as np

def simulate_pnl_path(params, n_trades, seed=None):
  p_win = params[0]
  win_amount = params[1]
  loss_amount = params[2]
  cost = params[3]

  rng = np.random.default_rng(seed)
  outcomes = (rng.random(n_trades) < p_win).astype(int)

  win_pnl = win_amount - cost
  loss_pnl = -loss_amount - cost

  trade_pnls = np.where(outcomes == 1, win_pnl, loss_pnl)
  cumulative_pnl = np.cumsum(trade_pnls)

  return outcomes, trade_pnls, cumulative_pnl

import numpy as np

def run_trials(params, n_trades, n_trials, seed=None):
  p_win = params[0]
  win_amount = params[1]
  loss_amount = params[2]
  cost = params[3]

  rng = np.random.default_rng(seed)

  outcomes = (rng.random((n_trials, n_trades)) < p_win).astype(int)

  win_pnl = win_amount - cost
  loss_pnl = -loss_amount - cost

  trade_pnls = np.where(outcomes == 1, win_pnl, loss_pnl)
  final_pnls = np.sum(trade_pnls, axis=1)

  mean_pnl = np.mean(final_pnls)
  ev_per_trade = p_win * win_amount - (1 - p_win) * loss_amount - cost
  theoretical_ev = ev_per_trade * n_trades

  return final_pnls, mean_pnl, theoretical_ev

## Detect and include frictions

**Slippage, Tail risk, and a full stress test in the end**

In [5]:
import numpy as np

def ev_with_slippage (params, avg_slippage):
  p_win = params[0]
  win_amount = params[1]
  loss_amount = params[2]
  cost = params[3]

  base_ev = p_win * win_amount - (1 - p_win) * loss_amount - cost
  adjusted_ev = base_ev - avg_slippage
  slippage_impact = -avg_slippage

  return base_ev, adjusted_ev, slippage_impact

def ev_with_tail_risk(params, tail_prob, tail_loss):
  p_win = params[0]
  win_amount = params[1]
  loss_amount = params[2]
  cost = params[3]

  base_ev = p_win * win_amount - (1 - p_win) * loss_amount - cost

  remaining = 1 - tail_prob
  adj_p_win = p_win * remaining
  adj_p_loss = (1 - p_win) * remaining

  adjusted_ev = (adj_p_win * (win_amount - cost)
                 + adj_p_loss * (-loss_amount - cost)
                 + tail_prob * (-tail_loss - cost))

  tail_impact = adjusted_ev - base_ev

  return base_ev, adjusted_ev, tail_impact

def stress_test(params, avg_slippage, tail_prob, tail_loss):
  p_win = params[0]
  win_amount = params[1]
  loss_amount = params[2]
  cost = params[3]

  base_ev = p_win * win_amount - (1 - p_win) * loss_amount - cost

  remaining = 1 - tail_prob
  adj_p_win = p_win * remaining
  adj_p_loss = (1 - p_win) * remaining

  ev_with_tail = (adj_p_win * (win_amount - cost)
                  + adj_p_loss * (-loss_amount - cost)
                  + tail_prob * (-tail_loss - cost))

  stressed_ev = ev_with_tail - avg_slippage
  survives = 1 if stressed_ev > 0 else 0

  return base_ev, stressed_ev, survives



## Final Decision

In [ ]:
import numpy as np

def trading_decision(params, avg_slippage, tail_prob, tail_loss):
  p_win = params[0]
  win_amount = params[1]
  loss_amount = params[2]
  cost = params[3]

  base_ev = p_win * win_amount - (1 - p_win) * loss_amount - cost

  remaining = 1 - tail_prob
  adj_p_win = p_win * remaining
  adj_p_loss = (1 - p_win) * remaining

  ev_with_tail = (adj_p_win * (win_amount - cost)
                  + adj_p_loss * (-loss_amount - cost)
                  + tail_prob * (-tail_loss - cost))

  stressed_ev = ev_with_tail - avg_slippage

  break_even_p = (loss_amount + cost) / (win_amount + loss_amount)
  margin = p_win - break_even_p

  verdict = 1 if stressed_ev > 0 else 0

  return verdict, base_ev, stressed_ev, margin

